# 02 — LoRA Rank Sweep on Banking77

Part of [bert-lora-finetuning](../README.md). Independent of `01_baseline...ipynb` — this notebook
loads its own data and only trains LoRA models, at several ranks. It doesn't need 01 to run, but the
accuracy plot is more useful with 01's full fine-tune number as a reference line.

**What this answers:** as rank `r` increases (more trainable parameters), how does accuracy change, and
where do returns start diminishing? This turns the single `r=8` number from notebook 01 into an actual
empirical curve instead of one point.

**Run on Colab or Kaggle with a GPU runtime.** This trains 4 separate LoRA models — expect roughly
4x the LoRA training time from notebook 01.

> **Kaggle: turn Internet on before running.** Sidebar → *Settings* → *Internet: On* (needs a
> phone-verified Kaggle account). With it off, pip and the HuggingFace Hub both fail with
> `Temporary failure in name resolution`, and pip then reports every package as
> `from versions: none` — that's a DNS failure, not a missing version. The preflight cell below
> checks this and tells you which mode you're in.

## Setup

In [ ]:
# --- Preflight: environment, network, results dir -------------------------------------------
# Run this first. It diagnoses the two things that break this notebook on hosted runtimes:
# no internet, and a non-writable results path.
import os
import socket
import sys

def _resolves(host: str) -> bool:
    try:
        socket.setdefaulttimeout(5)
        socket.gethostbyname(host)
        return True
    except OSError:
        return False

ON_KAGGLE = os.path.isdir("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
ONLINE = _resolves("pypi.org")

# Only /kaggle/working is writable on Kaggle — the repo-relative "../results" resolves to
# /kaggle/results there and raises PermissionError on makedirs.
if ON_KAGGLE:
    RESULTS_DIR = "/kaggle/working/results"
elif ON_COLAB:
    RESULTS_DIR = "/content/results"
else:
    RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Environment : {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'local'}")
print(f"Network     : {'online' if ONLINE else 'OFFLINE'}")
print(f"Results dir : {RESULTS_DIR}")

if not ONLINE:
    print(
        "\n>>> No DNS. On Kaggle: sidebar -> Settings -> Internet: On (needs phone verification).\n"
        ">>> Without it, pip and the HuggingFace Hub are both unreachable. You can still run\n"
        ">>> offline IF you attach Banking77 and bert-base-uncased as Kaggle Inputs and set\n"
        ">>> KAGGLE_DATA_DIR / MODEL_NAME to those paths in the data cell below."
    )

# Paste the full-fine-tune accuracy printed at the end of 01_baseline_full_finetune_vs_lora.ipynb here.
# Leave as None to skip the reference line on the accuracy plot.
FULL_FINETUNE_ACCURACY = None  # e.g. 0.9123

In [ ]:
# --- Install only what's actually missing ---------------------------------------------------
# Kaggle and Colab already ship transformers, datasets, torch and scikit-learn. Two rules:
#   1. Never pip-install torch here — it replaces the runtime's CUDA-matched build and can
#      silently drop you to CPU or break the driver match.
#   2. Don't reinstall packages that already import — that's what pulls in the datasets 4.x
#      churn people work around with pins like "datasets<4.0".
import importlib.util
import subprocess
import sys

REQUIRED = {           # import name -> pip name
    "transformers": "transformers",
    "datasets": "datasets",
    "peft": "peft",
    "accelerate": "accelerate",
    "sklearn": "scikit-learn",
}

missing = [pip_name for mod, pip_name in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if not missing:
    print("All required packages already present — nothing to install.")
elif not ONLINE:
    raise RuntimeError(
        f"Missing {missing} and there is no network. Turn Internet on (see the preflight cell)."
    )
else:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import importlib.metadata as md
for mod in REQUIRED:
    try:
        print(f"{mod:<14} {md.version(REQUIRED[mod])}")
    except md.PackageNotFoundError:
        print(f"{mod:<14} NOT INSTALLED")

In [ ]:
import gc
import inspect
import json
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected — this notebook trains 4 models, CPU will take a long time.")
    print("         Kaggle: sidebar -> Settings -> Accelerator -> GPU T4 x2 (or P100).")
else:
    print("GPU:", torch.cuda.get_device_name(0))

# `evaluate.load("accuracy")` fetches a metric script from the Hub at call time — one more
# network dependency, and it fails offline. sklearn computes the same two numbers locally.

# TrainingArguments renamed evaluation_strategy -> eval_strategy in transformers 4.41.
# Kaggle images lag; resolve the name instead of hard-coding it.
_TA_PARAMS = inspect.signature(TrainingArguments.__init__).parameters
EVAL_STRATEGY_KW = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"
print("TrainingArguments eval kwarg:", EVAL_STRATEGY_KW)

## Data

In [ ]:
MODEL_NAME = "bert-base-uncased"
DATASET_NAME = "PolyAI/banking77"
KAGGLE_DATA_DIR = "/kaggle/input/banking77"   # only used when offline; point at your attached Input

def load_banking77():
    """Hub when online; a CSV copy attached as a Kaggle Input when not."""
    if ONLINE:
        return load_dataset(DATASET_NAME)

    import glob
    if not os.path.isdir(KAGGLE_DATA_DIR):
        raise RuntimeError(
            "No network and no local Banking77 copy.\n"
            "Fix one of these:\n"
            "  (a) Sidebar -> Settings -> Internet: On (needs phone verification), or\n"
            f"  (b) attach Banking77 as a Kaggle Input and set KAGGLE_DATA_DIR (now: {KAGGLE_DATA_DIR})."
        )
    files = {}
    for split in ("train", "test"):
        hits = glob.glob(f"{KAGGLE_DATA_DIR}/**/*{split}*.csv", recursive=True)
        if not hits:
            raise RuntimeError(f"No '{split}' CSV found under {KAGGLE_DATA_DIR}")
        files[split] = hits[0]
    print("Loading offline from:", files)
    return load_dataset("csv", data_files=files)

dataset = load_banking77()

# Normalise the label column: the Hub version has an int ClassLabel called "label";
# CSV exports usually have a string column called "category" or "label".
label_col = "label" if "label" in dataset["train"].column_names else "category"
feature = dataset["train"].features[label_col]
if not hasattr(feature, "names"):
    dataset = dataset.class_encode_column(label_col)
if label_col != "label":
    dataset = dataset.rename_column(label_col, "label")

label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"Number of intent classes: {num_labels}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_ds = tokenized["train"]
eval_ds = tokenized["test"]
print(f"Train size: {len(train_ds)}, Test size: {len(eval_ds)}")

## Shared helpers

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def peak_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None

## Sweep: train LoRA at r = 4, 8, 16, 32

In [ ]:
RANKS = [4, 8, 16, 32]
ALPHA_TO_RANK_RATIO = 2  # keep alpha/r scaling constant (alpha = 2r) across the sweep

sweep_results = []

for r in RANKS:
    print(f"\n=== Training LoRA with r={r} ===")

    base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=r,
        lora_alpha=r * ALPHA_TO_RANK_RATIO,
        lora_dropout=0.1,
        target_modules=["query", "value"],
    )
    model = get_peft_model(base_model, lora_config)
    model.to(device)

    trainable, total = count_trainable_params(model)

    args = TrainingArguments(
        output_dir=f"{RESULTS_DIR}/lora_sweep_r{r}",
        learning_rate=2e-4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=4,
        save_strategy="no",
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        report_to="none",
        **{EVAL_STRATEGY_KW: "epoch"},
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
    )

    reset_peak_memory()
    start = time.time()
    trainer.train()
    train_time = time.time() - start
    peak_mem = peak_memory_mb()

    eval_results = trainer.evaluate()

    sweep_results.append({
        "rank": r,
        "alpha": r * ALPHA_TO_RANK_RATIO,
        "trainable_params": trainable,
        "trainable_pct": round(100 * trainable / total, 3),
        "accuracy": eval_results["eval_accuracy"],
        "macro_f1": eval_results["eval_macro_f1"],
        "train_time_s": round(train_time, 1),
        "peak_mem_mb": round(peak_mem, 1) if peak_mem else None,
    })

    print(f"r={r}: acc={eval_results['eval_accuracy']:.4f}, "
          f"trainable={trainable:,} ({100*trainable/total:.2f}%), time={train_time:.1f}s")

    # free memory before the next rank
    del model, base_model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Checkpoint after every rank: a 4x-length run that dies at r=32 should not lose r=4..16.
    pd.DataFrame(sweep_results).to_csv(f"{RESULTS_DIR}/02_rank_sweep.csv", index=False)

sweep_df = pd.DataFrame(sweep_results)
sweep_df

In [ ]:
sweep_df.to_csv(f"{RESULTS_DIR}/02_rank_sweep.csv", index=False)
with open(f"{RESULTS_DIR}/02_rank_sweep.json", "w") as f:
    json.dump(sweep_df.to_dict(orient="records"), f, indent=2)
print(f"Saved results to {RESULTS_DIR}/02_rank_sweep.{{csv,json}}")
if ON_KAGGLE:
    print("On Kaggle these land in /kaggle/working — download them from the Output panel and")
    print("commit them into the repo's results/ directory so the README and dashboard pick them up.")

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(sweep_df["rank"], sweep_df["accuracy"], marker="o", color="#55A868", label="LoRA")
if FULL_FINETUNE_ACCURACY is not None:
    axes[0].axhline(FULL_FINETUNE_ACCURACY, color="#4C72B0", linestyle="--", label="Full fine-tune")
axes[0].set_xlabel("Rank (r)")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy vs. Rank")
axes[0].set_xscale("log", base=2)
axes[0].legend()

axes[1].plot(sweep_df["rank"], sweep_df["trainable_params"], marker="o", color="#C44E52")
axes[1].set_xlabel("Rank (r)")
axes[1].set_ylabel("Trainable Parameters")
axes[1].set_title("Trainable Parameters vs. Rank")
axes[1].set_xscale("log", base=2)
axes[1].set_yscale("log")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/02_rank_sweep.png", dpi=150)
plt.show()

**Read the curve, don't just eyeball it:** where does accuracy stop improving as `r` grows? That point
is the actual rank/performance trade-off — note it explicitly, with numbers, in the README.
